In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub
from datetime import datetime
from scipy import stats

# 1. Download the complete dataset directory (returns the local path)
dataset_path = kagglehub.dataset_download("kartik2112/fraud-detection")
print("Dataset downloaded to:", dataset_path)

# 2. Define the individual file paths inside that directory
train_file_path = os.path.join(dataset_path, "fraudTrain.csv")
test_file_path = os.path.join(dataset_path, "fraudTest.csv")

# 3. Load them into separate pandas DataFrames
print("Loading training set...")
df_train = pd.read_csv(train_file_path)

print("Loading test set...")
df_test = pd.read_csv(test_file_path)

# 4. Verify the shapes
print(f"\nTraining set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")

Dataset downloaded to: C:\Users\charc\.cache\kagglehub\datasets\kartik2112\fraud-detection\versions\1
Loading training set...
Loading test set...

Training set shape: (1296675, 23)
Test set shape: (555719, 23)


## Data Preprocessing

In [5]:
df_train.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [6]:
# Merge the two dataframes for preprocessing & feature scaling
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

In [7]:
# Remove the unnecessary column
df = df.drop(columns=('Unnamed: 0'))

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 22 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   trans_date_trans_time  str    
 1   cc_num                 int64  
 2   merchant               str    
 3   category               str    
 4   amt                    float64
 5   first                  str    
 6   last                   str    
 7   gender                 str    
 8   street                 str    
 9   city                   str    
 10  state                  str    
 11  zip                    int64  
 12  lat                    float64
 13  long                   float64
 14  city_pop               int64  
 15  job                    str    
 16  dob                    str    
 17  trans_num              str    
 18  unix_time              int64  
 19  merch_lat              float64
 20  merch_long             float64
 21  is_fraud               int64  
dtypes: float64(5), int64(5), str(

In [9]:
# Fix data types of date/datetime features
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob']).dt.date

In [10]:
# Check null values
df.isnull().sum()

trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [11]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 22 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   trans_date_trans_time  datetime64[us]
 1   cc_num                 int64         
 2   merchant               str           
 3   category               str           
 4   amt                    float64       
 5   first                  str           
 6   last                   str           
 7   gender                 str           
 8   street                 str           
 9   city                   str           
 10  state                  str           
 11  zip                    int64         
 12  lat                    float64       
 13  long                   float64       
 14  city_pop               int64         
 15  job                    str           
 16  dob                    object        
 17  trans_num              str           
 18  unix_time              int64     

In [13]:
# Fixing the 'merchant' feature
df['merchant'] = df['merchant'].str.removeprefix('fraud_')

In [14]:
# Split the columns by data type
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=str).columns.tolist()

## Exploratory Data Analysis

In [15]:
# Checking proportion of fraud vs non-fraud transactions
fraud_rate = (df['is_fraud'].value_counts(normalize=True) * 100)
print(f"Total fraud rate: {fraud_rate.loc[1]:.2f}%")

Total fraud rate: 0.52%


### Univariate Analysis

In [19]:
# Check Skewness & Kurtosis
print('Skewness:')
for col in num_cols:
    print(col, ':', df[col].skew().round(2))

print('\nKurtosis:')
for col in num_cols:
    print(col, ':', df[col].kurt().round(2))

Skewness:
cc_num : 2.85
amt : 40.81
zip : 0.08
lat : -0.19
long : -1.15
city_pop : 5.59
unix_time : -0.02
merch_lat : -0.19
merch_long : -1.14
is_fraud : 13.75

Kurtosis:
cc_num : 6.18
amt : 4181.91
zip : -1.1
lat : 0.79
long : 1.84
city_pop : 37.57
unix_time : -1.2
merch_lat : 0.77
merch_long : 1.83
is_fraud : 186.94


In [ ]:
# Shapiro-Wilk test to check the normality
stat, p = stats.shapiro(df[col].dropna().sample(min(5000, len(df))))
print('Normal' if p > 0.05 else 'Not normal')

Not normal


In [17]:
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [18]:
df.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud'],
      dtype='str')

# ---Work in progress---